# Notebook 07 (No Augmentation) — Final Benchmarking and Evaluation

This notebook loads the per-fold metrics saved by the no-augmentation variants of
notebooks 05 and 06 (`revised_no_augmentation/`) and performs the final benchmarking
comparison of PLS-DA, SVM, and 1D-CNN trained **without data augmentation**.

No models are retrained here — only saved results are loaded and analyzed.

## Section 1 — Imports and Setup

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2
from sklearn.metrics import confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11

os.makedirs('../../docs/figures', exist_ok=True)
os.makedirs('../../data/results/no_aug', exist_ok=True)

print('Setup complete.')

## Section 2 — Load Saved Results

Per-fold metrics are loaded from `data/results/no_aug/` — the JSON files saved by
the no-augmentation training notebooks.

In [ ]:
with open('../../data/results/no_aug/plsda_results.json') as f:
    plsda = json.load(f)

with open('../../data/results/no_aug/svm_results.json') as f:
    svm = json.load(f)

with open('../../data/results/no_aug/cnn_results.json') as f:
    cnn = json.load(f)

print('Results loaded successfully.')
print(f'PLS-DA : {len(plsda["fold_metrics"])} folds')
print(f'SVM    : {len(svm["fold_metrics"])} folds')
print(f'1D-CNN : {len(cnn["fold_metrics"])} folds')

def get_f1(m):
    return m.get('f1_score', m.get('f1', 0.0))

## Section 3 — Summary of Mean Metrics Across 5 Folds

In [ ]:
rows = []
for label, res in [('PLS-DA', plsda), ('SVM', svm), ('1D-CNN', cnn)]:
    rows.append({
        'Model':     label,
        'Accuracy':  f"{res['mean_accuracy']:.4f} +/- {res['std_accuracy']:.4f}",
        'Precision': f"{res['mean_precision']:.4f} +/- {res['std_precision']:.4f}",
        'Recall':    f"{res['mean_recall']:.4f} +/- {res['std_recall']:.4f}",
        'F1-Score':  f"{res['mean_f1']:.4f} +/- {res['std_f1']:.4f}",
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

best_by_f1 = max(
    [('PLS-DA', plsda), ('SVM', svm), ('1D-CNN', cnn)],
    key=lambda x: x[1]['mean_f1']
)[0]
print(f'\nHighest mean F1-Score: {best_by_f1}')

## Section 4 — Per-Fold Metrics

In [ ]:
fold_rows = []
for label, res in [('PLS-DA', plsda), ('SVM', svm), ('1D-CNN', cnn)]:
    for i, m in enumerate(res['fold_metrics']):
        fold_rows.append({
            'Model':     label,
            'Fold':      i + 1,
            'Accuracy':  round(m['accuracy'],  4),
            'Precision': round(m['precision'], 4),
            'Recall':    round(m['recall'],    4),
            'F1-Score':  round(get_f1(m),      4),
        })

fold_df = pd.DataFrame(fold_rows)
display(fold_df)

## Section 5 — Model Performance Comparison

In [ ]:
metrics     = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_keys = [('mean_accuracy', 'std_accuracy'),
               ('mean_precision', 'std_precision'),
               ('mean_recall', 'std_recall'),
               ('mean_f1', 'std_f1')]

models_data = {
    'PLS-DA': {
        'means':  [plsda[mk] for mk, _ in metric_keys],
        'stds':   [plsda[sk] for _, sk in metric_keys],
        'color':  'steelblue',
    },
    'SVM': {
        'means':  [svm[mk] for mk, _ in metric_keys],
        'stds':   [svm[sk] for _, sk in metric_keys],
        'color':  'tomato',
    },
    '1D-CNN': {
        'means':  [cnn[mk] for mk, _ in metric_keys],
        'stds':   [cnn[sk] for _, sk in metric_keys],
        'color':  'mediumseagreen',
    },
}

x         = np.arange(len(metrics))
bar_width = 0.25
offsets   = [-bar_width, 0, bar_width]

fig, ax = plt.subplots(figsize=(12, 6))

for (name, data), offset in zip(models_data.items(), offsets):
    bars = ax.bar(
        x + offset, data['means'], bar_width,
        yerr=data['stds'], capsize=4,
        color=data['color'], alpha=0.85,
        error_kw={'ecolor': 'gray', 'elinewidth': 1},
        label=name,
    )
    for bar, mean_val in zip(bars, data['means']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{mean_val:.3f}',
            ha='center', va='bottom', fontsize=8,
        )

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — 5-Fold CV (No Augmentation, n=80)')
ax.legend()
plt.tight_layout()
plt.savefig('../../docs/figures/comparison_bar_chart_no_aug.png', bbox_inches='tight')
plt.show()
print('Saved: docs/figures/comparison_bar_chart_no_aug.png')

## Section 6 — Combined Confusion Matrices

In [ ]:
y_true_plsda = np.array(plsda['all_y_true'])
y_pred_plsda = np.array(plsda['all_y_pred'])
y_true_svm   = np.array(svm['all_y_true'])
y_pred_svm   = np.array(svm['all_y_pred'])
y_true_cnn   = np.array(cnn['all_y_true'])
y_pred_cnn   = np.array(cnn['all_y_pred'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_data = [
    ('PLS-DA', y_true_plsda, y_pred_plsda),
    ('SVM',    y_true_svm,   y_pred_svm),
    ('1D-CNN', y_true_cnn,   y_pred_cnn),
]

for ax, (name, y_true, y_pred) in zip(axes, plot_data):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Low Protein', 'High Protein'],
        yticklabels=['Low Protein', 'High Protein'],
    )
    ax.set_title(name)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig.suptitle('Confusion Matrices — 5-Fold CV (No Augmentation, n=80)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../../docs/figures/confusion_matrices_no_aug.png', bbox_inches='tight')
plt.show()
print('Saved: docs/figures/confusion_matrices_no_aug.png')

## Section 7 — McNemar's Test for Statistical Significance

In [ ]:
def mcnemar_test(y_true, y_pred_a, y_pred_b):
    y_true   = np.array(y_true)
    y_pred_a = np.array(y_pred_a)
    y_pred_b = np.array(y_pred_b)

    b = np.sum((y_pred_a == y_true) & (y_pred_b != y_true))
    c = np.sum((y_pred_a != y_true) & (y_pred_b == y_true))

    if (b + c) == 0:
        return 0.0, 1.0

    statistic = (abs(b - c) - 1) ** 2 / (b + c)
    p_value   = 1 - chi2.cdf(statistic, df=1)
    return float(statistic), float(p_value)


pairs = [
    ('PLS-DA', 'SVM',    y_true_plsda, y_pred_plsda, y_pred_svm),
    ('PLS-DA', '1D-CNN', y_true_plsda, y_pred_plsda, y_pred_cnn),
    ('SVM',    '1D-CNN', y_true_svm,   y_pred_svm,   y_pred_cnn),
]

mcnemar_results = []
for name_a, name_b, y_true, y_pred_a, y_pred_b in pairs:
    stat, p     = mcnemar_test(y_true, y_pred_a, y_pred_b)
    significant = 'Yes' if p < 0.05 else 'No'
    mcnemar_results.append({
        'Comparison':           f'{name_a} vs {name_b}',
        'Statistic':            round(stat, 4),
        'p-value':              round(p, 4),
        'Significant (p<0.05)': significant,
    })
    p_display = f'{p:.4e}' if p < 0.0001 else f'{p:.4f}'
    print(f'{name_a} vs {name_b}: statistic={stat:.4f}  p={p_display}  significant={significant}')

mcnemar_df = pd.DataFrame(mcnemar_results)
display(mcnemar_df)

## Section 8 — Best Model Selection

In [ ]:
models_summary = {
    'PLS-DA': plsda,
    'SVM':    svm,
    '1D-CNN': cnn,
}

best_model_name = max(models_summary, key=lambda k: models_summary[k]['mean_f1'])
best_model      = models_summary[best_model_name]

print('=' * 56)
print('  Best Model Selection (No Augmentation)')
print('=' * 56)
for name, res in models_summary.items():
    marker = ' <-- BEST' if name == best_model_name else ''
    print(f'  {name:10s} | F1: {res["mean_f1"]:.4f} +/- {res["std_f1"]:.4f}{marker}')
print('=' * 56)
print(f'\nSelected best model : {best_model_name}')
print(f'Mean F1-Score       : {best_model["mean_f1"]:.4f} +/- {best_model["std_f1"]:.4f}')
print(f'Mean Accuracy       : {best_model["mean_accuracy"]:.4f} +/- {best_model["std_accuracy"]:.4f}')

## Section 9 — Save Final Results

In [ ]:
summary_df.to_csv('../../data/results/no_aug/final_comparison.csv', index=False)
print('Saved: data/results/no_aug/final_comparison.csv')

mcnemar_df.to_csv('../../data/results/no_aug/mcnemar_results.csv', index=False)
print('Saved: data/results/no_aug/mcnemar_results.csv')

with open('../../data/results/no_aug/best_model.txt', 'w') as f:
    f.write(best_model_name)
print(f'Saved: data/results/no_aug/best_model.txt  ({best_model_name})')

## Section 10 — Summary

| Item | Detail |
|---|---|
| **Models compared** | PLS-DA, SVM, 1D-CNN |
| **Augmentation** | None — all models trained on raw fold data only |
| **Evaluation method** | 5-Fold Stratified Cross Validation — all 80 original samples evaluated exactly once |
| **Primary metric** | F1-Score |
| **Results location** | `data/results/no_aug/` |
| **Figures** | Saved to `docs/figures/` with `_no_aug` suffix |

### Key notes
- Results from this notebook can be directly compared with `revised/07_evaluation_metrics.ipynb` to quantify the impact of data augmentation on model performance.
- The `+/-` notation is used throughout in place of the Unicode plus-minus sign.